In [4]:
import pandas as pd

RAW_DATA_PATH = 'en.openfoodfacts.org.products.csv'
SUBSET_DATA_PATH = 'subset_snacks.csv'

# Only load the columns we actually need for the analysis
COLS_TO_KEEP = [
    'product_name',
    'categories_tags',
    'sugars_100g',
    'proteins_100g',
    'ingredients_text'
]

print("Loading data...")

df = pd.read_csv(
    RAW_DATA_PATH,
    sep='\t',
    usecols=COLS_TO_KEEP,
    nrows=500000,
    low_memory=False,
    on_bad_lines='skip'     # Silently drops any corrupted rows
)

print(f"Successfully loaded {len(df)} rows!")
df.to_csv(SUBSET_DATA_PATH, index=False)
df.head()

Loading data...
Successfully loaded 500000 rows!


,product_name,categories_tags,ingredients_text,sugars_100g,proteins_100g
0,Limonade artisanale a la rose,NaN,NaN,NaN,NaN
1,M&amp;M white,NaN,"Weizenmehl, Rapsöl, Speisesalz, 1,7% Meersalz,...",NaN,NaN
2,Chocolate n3,NaN,NaN,NaN,NaN
3,Pâte de fruits,NaN,NaN,NaN,NaN
4,Paleta gran reserva - Sierra nevada-,"en:beverages-and-beverages-preparations,en:bev...","Thiamin, Biotin, Chromium, Garcinia cambogia f...",NaN,NaN


In [6]:
print(f"Rows before cleaning: {len(df)}")

# 1. Drop rows missing critical data
df_clean = df.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g']).copy()

# 2. Filter out impossible outliers (Values must be between 0 and 100 per 100g)
df_clean = df_clean[
    (df_clean['sugars_100g'] >= 0) & (df_clean['sugars_100g'] <= 100) &
    (df_clean['proteins_100g'] >= 0) & (df_clean['proteins_100g'] <= 100)
]

print(f"Rows after cleaning: {len(df_clean)}")
print(f"Dropped {len(df) - len(df_clean)} invalid rows.")

# 3. Save this clean baseline
CLEANED_DATA_PATH = 'cleaned_snacks.csv'
df_clean.to_csv(CLEANED_DATA_PATH, index=False)
print(f"Saved clean dataset to {CLEANED_DATA_PATH}")

df_clean.describe()

Rows before cleaning: 500000
Rows after cleaning: 108087
Dropped 391913 invalid rows.
Saved clean dataset to cleaned_snacks.csv


,sugars_100g,proteins_100g
count,108087.000000,108087.000000
mean,12.852077,8.297599
std,18.475758,9.949222
min,0.000000,0.000000
25%,1.009158,2.000000
50%,4.650000,6.000000
75%,16.000000,10.700000
max,100.000000,100.000000
